# Faz 6 — Backbone Ablation (2D Sınıflandırma)

**Amaç:** Aynı NPZ verisi, aynı hiperparametreler, farklı backbone → mAP/macroF1 karşılaştırması.

| # | Backbone | Tür | Pretraining | Çözünürlük |
|---|----------|-----|-------------|------------|
| 1 | `swin_base` | Transformer | IN-22k → IN-1k | 384 |
| 2 | `convnext_base` | CNN | IN-22k → IN-1k | 384 |
| 3 | `convnextv2_base` | CNN+MAE | IN-22k → IN-1k | 384 |
| 4 | `vit_base_16` | ViT | IN-21k AugReg | 384 |
| 5 | `dinov2_base` | ViT | Self-supervised | 448 |
| 6 | `eva02_base` | ViT+MIM | IN-22k | 448 |

**Kullanım:** `BACKBONE_KEY` değiştir → hücreleri çalıştır → tekrarla.  
Son hücre tüm kaydedilmiş sonuçları karşılaştırır.


---
## 0. Ortam

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f"Ortam : {env_name}")

import torch
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif IS_LOCAL and torch.backends.mps.is_available():
    info = torch.mps.driver_allocated_memory
    print(f"GPU   : Apple Silicon MPS aktif")
    print(f"PyTorch: {torch.__version__}")
else:
    print("UYARI: GPU bulunamadı — CPU ile eğitim çok yavaş olacak")


In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata, drive
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        drive.mount('/content/drive', force_remount=False)
        print("Kaggle kimlik bilgileri + Drive bağlandı")
    except Exception as e:
        print(f"Manuel kurulum gerekebilir: {e}")
else:
    print(f"{env_name} — Colab kurulumu atlandı")


---
## 1. Kütüphane Kurulumu

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'timm>=0.9.16', 'torch', 'torchvision', 'pandas', 'scikit-learn', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()
import timm
print(f"timm  : {timm.__version__}")
print(f"torch : {torch.__version__}")


---
## 2. Backbone Seçimi

Sadece `BACKBONE_KEY` değiştir ve çalıştır.  
Her backbone için checkpoint ve sonuç ayrı klasöre kaydedilir.


In [ ]:
import json, shutil, time
import numpy as np
import pandas as pd
from typing import Dict

# ── Ablasyon backbone kataloğu ─────────────────────────────────────────────
# (backbone_timm_adı, input_size, batch_size, notlar)
BACKBONE_CATALOG = {
    'swin_base': (
        'swin_base_patch4_window12_384.ms_in22k_ft_in1k',
        384, 16,
        'SwinTransformer-B  |  baseline  |  IN-22k pretrain'
    ),
    'convnext_base': (
        'convnext_base.fb_in22k_ft_in1k_384',
        384, 16,
        'ConvNeXt-B  |  CNN  |  IN-22k pretrain'
    ),
    'convnextv2_base': (
        'convnextv2_base.fcmae_ft_in22k_in1k_384',
        384, 16,
        'ConvNeXt-V2-B  |  CNN+MAE  |  IN-22k pretrain'
    ),
    'vit_base_16': (
        'vit_base_patch16_384.augreg_in21k_ft_in1k',
        384, 16,
        'ViT-B/16  |  plain ViT  |  AugReg IN-21k pretrain'
    ),
    'dinov2_base': (
        'vit_base_patch14_reg4_dinov2.lvd142m',
        448, 10,
        'DINOv2-B  |  self-supervised  |  LVD-142M'
    ),
    'eva02_base': (
        'eva02_base_patch14_448.mim_in22k_ft_in22k_in1k',
        448, 10,
        'EVA-02-B  |  MIM  |  IN-22k pretrain'
    ),
}

# ─── BURADAN DEĞİŞTİR ────────────────────────────────────────────────────
BACKBONE_KEY = 'convnext_base'   # swin_base | convnext_base | convnextv2_base
                                  # vit_base_16 | dinov2_base | eva02_base
FOLD        = 0
GITHUB_URL  = 'https://github.com/ramazan2020/abdomen1.git'
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
# ─────────────────────────────────────────────────────────────────────────

assert BACKBONE_KEY in BACKBONE_CATALOG, f"Geçersiz BACKBONE_KEY: {BACKBONE_KEY}"
_bb_timm, _input_size, _batch_size, _bb_note = BACKBONE_CATALOG[BACKBONE_KEY]

print(f"Backbone : {BACKBONE_KEY}")
print(f"  timm   : {_bb_timm}")
print(f"  size   : {_input_size}×{_input_size}")
print(f"  batch  : {_batch_size}")
print(f"  not    : {_bb_note}")
print(f"Fold     : {FOLD}")


---
## 3. Yollar ve Ortam Değişkenleri

In [ ]:
from pathlib import Path

if IS_KAGGLE:
    DATA_DIR   = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR   = Path('/kaggle/working')
elif IS_COLAB:
    DATA_DIR   = Path('/content/kaggle_data')
    DRIVE_BASE = Path('/content/drive/MyDrive/Abdomen')
    WORK_DIR   = Path('/content/abdomen_work')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))

EGITIM_DIR  = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(DATA_DIR / 'Egitim Verisi')))
YARISMA_DIR = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(DATA_DIR / 'Yarisma Verisi')))
SPLIT_DIR   = Path(os.environ.get('ABDOMEN_SPLIT_DIR', str(WORK_DIR / 'splits'))) if IS_LOCAL else (DATA_DIR / 'outputs' / 'splits')
NPZ_DIR     = Path(os.environ.get('ABDOMEN_CLS_DATA_DIR', str(WORK_DIR / 'cls_data')))

# Backbone'a özel checkpoint ve log dizini
CKPT_DIR    = WORK_DIR / f'checkpoints_ablation/{BACKBONE_KEY}'
LOG_DIR     = WORK_DIR / f'logs_ablation/{BACKBONE_KEY}'
ABLATION_DIR= WORK_DIR / 'ablation_results'

for d in [NPZ_DIR, CKPT_DIR, LOG_DIR, ABLATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# src/config.py import öncesi env var'ları ayarla
os.environ['ABDOMEN_SPLIT_DIR']    = str(SPLIT_DIR)
os.environ['ABDOMEN_CLS_DATA_DIR'] = str(NPZ_DIR)
os.environ['ABDOMEN_CKPT_DIR']     = str(CKPT_DIR)
os.environ['ABDOMEN_LOG_DIR']      = str(LOG_DIR)
os.environ['ABDOMEN_TRAIN_DIR']    = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']     = str(YARISMA_DIR)
os.environ['ABDOMEN_OUT_DIR']      = str(WORK_DIR / 'abdomen_src_out')

print(f"SPLIT_DIR   : {SPLIT_DIR}  ({'✓' if SPLIT_DIR.exists() else '✗'})")
print(f"NPZ_DIR     : {NPZ_DIR}  ({'✓' if NPZ_DIR.exists() else '✗ — önce Faz4 çalıştır'})")
print(f"CKPT_DIR    : {CKPT_DIR}")
print(f"LOG_DIR     : {LOG_DIR}")
print(f"ABLATION_DIR: {ABLATION_DIR}")


In [ ]:
# ── GitHub Repo / Local src ───────────────────────────────────────────────
if IS_LOCAL:
    REPO_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    print(f'Local: src/ → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(['git','clone','--depth=1', GITHUB_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.config    import ClsConfig, SUPER_CLASSES
from src.splits    import load_fold
from src.train_cls import train_one_fold, evaluate, build_model
from src.datasets  import SliceMultiLabelDataset

print('src modülleri ✓')

# Ablasyon için ClsConfig oluştur
abl_cfg = ClsConfig(
    backbone      = _bb_timm,
    input_size    = _input_size,
    batch_size    = _batch_size,
    num_classes   = 6,
    epochs        = 50,
    lr            = 2e-4,
    weight_decay  = 1e-4,
    warmup_epochs = 3,
    use_focal            = True,
    focal_gamma          = 2.0,
    use_weighted_sampler = True,
    mixup_alpha   = 0.2,
    accum_steps   = 1,
)

val_cases = load_fold(FOLD, 'val')
print(f'Val cases: {len(val_cases)}')

# Backbone'un timm'de mevcut olduğunu doğrula
import timm as _timm_check
try:
    _m = _timm_check.create_model(_bb_timm, pretrained=False, num_classes=6)
    _nparams = sum(p.numel() for p in _m.parameters()) / 1e6
    del _m
    print(f'Backbone doğrulandı: {_bb_timm}  ({_nparams:.1f}M parametre)')
except Exception as e:
    print(f'UYARI: Backbone yüklenemedi: {e}')
    print('timm sürümünü kontrol edin veya backbone adını düzeltin')


---
## 4. NPZ Verisi Kontrolü

NPZ dosyaları Faz4_SwinTransformer notebook'u tarafından üretilir.
Burada sadece varlık kontrolü yapılır.

In [ ]:
npz_files = list(NPZ_DIR.glob('*.npz'))
print(f'NPZ dosyası: {len(npz_files)}')

if len(npz_files) == 0:
    if IS_KAGGLE:
        # Kaggle'da NPZ bir dataset olarak eklenmiş olabilir
        _npz_input = Path('/kaggle/input/abdomen-npz')
        if _npz_input.exists():
            print('NPZ input dataset mevcut, kopyalanıyor...')
            import shutil
            for f in _npz_input.glob('*.npz'):
                shutil.copy2(str(f), str(NPZ_DIR / f.name))
            print(f'{len(list(NPZ_DIR.glob("*.npz")))} NPZ kopyalandı')
        else:
            print('HATA: NPZ bulunamadı!')
            print('Önce Faz4_SwinTransformer_Colab_Kaggle.ipynb çalıştırın (cell_npz_export)')
    else:
        raise FileNotFoundError(
            f'NPZ bulunamadı: {NPZ_DIR}\n'
            'Önce Faz4_SwinTransformer_Colab_Kaggle.ipynb cell_npz_export çalıştırın'
        )
else:
    import numpy as np
    _sample = np.load(str(npz_files[0]))
    print(f'  Örnek: {npz_files[0].name}  → keys: {list(_sample.keys())}')
    if 'image' in _sample:
        print(f'  image shape: {_sample["image"].shape}')
    print('NPZ verileri hazır ✓')


---
## 5. Eğitim

Checkpoint: `checkpoints_ablation/{BACKBONE_KEY}/cls_fold{FOLD}/best.pth`  
TensorBoard: `logs_ablation/{BACKBONE_KEY}/tb/cls_fold{FOLD}/`


In [ ]:
_ckpt_path = CKPT_DIR / f'cls_fold{FOLD}' / 'best.pth'

if _ckpt_path.exists():
    print(f'Checkpoint zaten mevcut: {_ckpt_path}')
    print('Yeniden eğitmek için checkpoint dosyasını silin.')
else:
    print(f'=== {BACKBONE_KEY} Eğitimi Başlıyor ===')
    print(f'  backbone   : {_bb_timm}')
    print(f'  input_size : {_input_size}')
    print(f'  batch_size : {_batch_size}')
    print(f'  epochs     : {abl_cfg.epochs}')
    print(f'  ckpt_dir   : {CKPT_DIR}')
    print('─' * 60)
    t0 = time.time()
    best = train_one_fold(fold=FOLD, cfg=abl_cfg)
    elapsed = time.time() - t0
    print(f'\nEğitim tamamlandı: {elapsed/3600:.1f} saat')
    print(f'Best mAP : {best.get("mAP", best.get("best_mAP", "?")):.4f}')

print(f'Checkpoint: {_ckpt_path}  ({"✓" if _ckpt_path.exists() else "✗"})')


---
## 6. Değerlendirme

In [ ]:
from torch.utils.data import DataLoader

_ckpt_path = CKPT_DIR / f'cls_fold{FOLD}' / 'best.pth'
if not _ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint bulunamadı: {_ckpt_path}\nÖnce eğitim hücresini çalıştırın.')

import torch
device = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))

model = build_model(abl_cfg).to(device)
state = torch.load(str(_ckpt_path), map_location=device)
model.load_state_dict(state['model'] if 'model' in state else state)
model.eval()
print(f'Model yüklendi: {_ckpt_path.name}')
print(f'Parametre: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

val_ds = SliceMultiLabelDataset(val_cases, input_size=_input_size)
val_loader = DataLoader(val_ds, batch_size=_batch_size, shuffle=False,
                        num_workers=4, pin_memory=(device.type=='cuda'))

metrics = evaluate(model, val_loader, device)
print()
print(f'=== {BACKBONE_KEY} — Fold {FOLD} Sonuçları ===')
print(f'mAP      : {metrics["mAP"]:.4f}')
print(f'macroF1  : {metrics["macroF1"]:.4f}')
print()
print(f'{"Sınıf":<33} {"AP":>7}  {"F1":>7}')
print('─' * 50)
for cls in SUPER_CLASSES:
    ap = metrics.get(f'AP/{cls}', float('nan'))
    f1 = metrics.get(f'F1/{cls}', float('nan'))
    print(f'  {cls:<31} {ap:>7.4f}  {f1:>7.4f}')


---
## 7. Sonucu Kaydet

In [ ]:
result = {
    'backbone_key' : BACKBONE_KEY,
    'backbone_timm': _bb_timm,
    'backbone_note': _bb_note,
    'input_size'   : _input_size,
    'batch_size'   : _batch_size,
    'fold'         : FOLD,
    'mAP'          : metrics['mAP'],
    'macroF1'      : metrics['macroF1'],
    'per_class'    : {cls: {'AP': metrics.get(f'AP/{cls}', float('nan')),
                            'F1': metrics.get(f'F1/{cls}', float('nan'))}
                     for cls in SUPER_CLASSES},
}

out_file = ABLATION_DIR / f'backbone_{BACKBONE_KEY}_fold{FOLD}.json'
out_file.write_text(json.dumps(result, indent=2, default=float))
print(f'Kaydedildi: {out_file}')


---
## 8. Tüm Backbone'ları Karşılaştır

Bu hücre, `ablation_results/` klasöründeki tüm `backbone_*_fold0.json` dosyalarını okur.
Henüz eğitilmemiş backbone'lar tabloda görünmez.


In [ ]:
import glob

result_files = sorted(ABLATION_DIR.glob(f'backbone_*_fold{FOLD}.json'))
if not result_files:
    print(f'Henüz sonuç yok: {ABLATION_DIR}')
else:
    results = [json.loads(f.read_text()) for f in result_files]
    results.sort(key=lambda r: r['mAP'], reverse=True)

    print(f'\n{"Backbone":<20} {"Input":>6} {"mAP":>7} {"macroF1":>8}  Not')
    print('─' * 75)
    for r in results:
        note_short = r.get('backbone_note','').split('|')[0].strip()
        marker = ' ← best' if r == results[0] else ''
        print(f'  {r["backbone_key"]:<18} {r["input_size"]:>6} '
              f'{r["mAP"]:>7.4f} {r["macroF1"]:>8.4f}  {note_short}{marker}')

    print()
    print('Sınıf bazında mAP:')
    cls_header = f'{"Backbone":<20}' + ''.join(f'{c[:8]:>10}' for c in SUPER_CLASSES)
    print(cls_header)
    print('─' * (20 + 10*len(SUPER_CLASSES)))
    for r in results:
        row = f'  {r["backbone_key"]:<18}'
        for cls in SUPER_CLASSES:
            ap = r['per_class'].get(cls, {}).get('AP', float('nan'))
            row += f'{ap:>10.4f}'
        print(row)

    # CSV kaydet
    rows = []
    for r in results:
        row = {'backbone': r['backbone_key'], 'input_size': r['input_size'],
               'mAP': r['mAP'], 'macroF1': r['macroF1']}
        for cls in SUPER_CLASSES:
            row[f'AP_{cls}'] = r['per_class'].get(cls,{}).get('AP', float('nan'))
            row[f'F1_{cls}'] = r['per_class'].get(cls,{}).get('F1', float('nan'))
        rows.append(row)
    df_abl = pd.DataFrame(rows)
    csv_path = ABLATION_DIR / f'backbone_ablation_fold{FOLD}.csv'
    df_abl.to_csv(csv_path, index=False)
    print(f'\nCSV: {csv_path}')
